# 🎬 Создание видео для маркетплейсов

**Инструкция:**
1. Запустите первую ячейку (установка библиотек).
2. Загрузите фото товара в папку `images` (иконка папки слева).
3. Запускайте нужные ячейки со скриптами по порядку.

In [ ]:
!pip install moviepy==1.0.3 opencv-python-headless Pillow numpy -q
print('✅ Библиотеки установлены!')

In [ ]:
import os
os.makedirs('images', exist_ok=True)
print('📁 Папка images готова. Перетащите ваши фото в неё через левую панель Colab.')

In [ ]:
# --- СКРИПТ 1: Базовая склейка ---
from moviepy.editor import ImageClip, concatenate_videoclips

IMAGE_FOLDER = "images"
OUTPUT_FILE = "output_basic.mp4"
DURATION = 3.0
SIZE = (1080, 1440)

files = sorted([os.path.join(IMAGE_FOLDER, f) for f in os.listdir(IMAGE_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
if files:
    clips = [ImageClip(f, duration=DURATION).resize(SIZE).set_position('center') for f in files]
    concatenate_videoclips(clips, method="compose").write_videofile(OUTPUT_FILE, fps=24, codec="libx264", audio=False, logger=None)
    print(f'✅ Готово: {OUTPUT_FILE}')
else:
    print('⚠️ Нет фото в папке images')

In [ ]:
# --- СКРИПТ 2: Плавные переходы ---
from moviepy.editor import ImageClip, CompositeVideoClip

OUTPUT_FILE = "output_crossfade.mp4"
FADE = 0.5

if files:
    clips = []
    for i, f in enumerate(files):
        c = ImageClip(f, duration=DURATION).resize(SIZE).set_position('center')
        c = c.set_start(0) if i == 0 else c.set_start(i * (DURATION - FADE)).crossfadein(FADE)
        clips.append(c)
    dur = len(clips) * DURATION - (len(clips) - 1) * FADE
    CompositeVideoClip(clips, size=SIZE).set_duration(dur).write_videofile(OUTPUT_FILE, fps=24, codec="libx264", audio=False, logger=None)
    print(f'✅ Готово: {OUTPUT_FILE}')

In [ ]:
# --- СКРИПТ 3: Эффект зума (Ken Burns) ---
OUTPUT_FILE = "output_zoom.mp4"
ZOOM = 1.15

if files:
    clips = []
    for f in files:
        base = ImageClip(f, duration=DURATION).resize((int(SIZE[0]*ZOOM), int(SIZE[1]*ZOOM)))
        anim = base.resize(lambda t: ZOOM - (ZOOM - 1.0) * (t / DURATION)).set_position('center')
        clips.append(anim)
    concatenate_videoclips(clips, method="compose").write_videofile(OUTPUT_FILE, fps=24, codec="libx264", audio=False, logger=None)
    print(f'✅ Готово: {OUTPUT_FILE}')

In [ ]:
# --- СКРИПТ 4: Адаптация формата (3:4, 1:1, 9:16) ---
OUTPUT_FILE = "output_adapted.mp4"
TARGET = (1080, 1440) # Поменяйте на (1080, 1080) для квадрата
BG = (245, 245, 245)

if files:
    clips = []
    for f in files:
        orig = ImageClip(f, duration=DURATION)
        scale = min(TARGET[0]/orig.w, TARGET[1]/orig.h)
        resized = orig.resize(scale)
        mx, my = (TARGET[0]-resized.w)/2, (TARGET[1]-resized.h)/2
        clips.append(resized.margin(left=mx, right=mx, top=my, bottom=my, color=BG).set_position('center'))
    CompositeVideoClip(clips, size=TARGET).set_duration(len(clips)*DURATION).write_videofile(OUTPUT_FILE, fps=24, codec="libx264", audio=False, logger=None)
    print(f'✅ Готово: {OUTPUT_FILE}')

In [ ]:
# --- СКРИПТ 5: Нарезка видео на фото ---
import cv2
VIDEO = "input_video.mp4" # Загрузите видео в корень
OUT_DIR = "frames"
COUNT = 5
os.makedirs(OUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO)
if cap.isOpened():
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = total // (COUNT + 1)
    for i in range(1, COUNT + 1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * interval)
        ret, frame = cap.read()
        if ret: cv2.imwrite(f"{OUT_DIR}/frame_{i:02d}.jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 100])
    cap.release()
    print(f'✅ Извлечено {COUNT} кадров в папку {OUT_DIR}')
else:
    print('⚠️ Видео не найдено. Загрузите input_video.mp4')

In [ ]:
# --- СКАЧИВАНИЕ РЕЗУЛЬТАТА ---
from google.colab import files
import glob
videos = glob.glob('output_*.mp4')
if videos:
    for v in videos: files.download(v)
else:
    print('Нет готовых видео. Запустите один из скриптов выше.')